# 03b. TabPFN on SBERT Embeddings

TabPFN v2 — transformer foundation model для табличных данных.
Применяется к SBERT embeddings из Phase 3.

**Лимиты:** max 10,000 строк, max 500 фичей.
** sklearn-совместим:** predict_proba для multiclass (до 10 классов).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import log_loss
from sklearn.model_selection import cross_val_predict
from pathlib import Path
import torch

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../output')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

train_df['target'] = (train_df['winner_model_a'].astype(int) * 0 +
                      train_df['winner_model_b'].astype(int) * 1 +
                      train_df['winner_tie'].astype(int) * 2)

# Load cached SBERT embeddings
X_train = np.load(OUTPUT_DIR / 'sbert_train_features.npy')
X_test = np.load(OUTPUT_DIR / 'sbert_test_features.npy')
y_train = train_df['target'].values

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

In [ ]:
# Check limits
n_samples, n_features = X_train.shape
print(f'Samples: {n_samples}, Features: {n_features}')

if n_samples > 10000:
    print('WARNING: samples exceed TabPFN v2 limit (10000). Subsampling needed.')
elif n_features > 500:
    print('WARNING: features exceed TabPFN v2 limit (500).')
else:
    print('Within TabPFN v2 limits.')

In [ ]:
# Install tabpfn if needed
# !uv pip install tabpfn

from tabpfn import TabPFNClassifier

clf = TabPFNClassifier(
    n_estimators=8,
    device=DEVICE,
    random_state=42
)

# OOF predictions via 5-fold CV
print('Running 5-fold CV...')
oof_preds = cross_val_predict(clf, X_train, y_train, cv=5, method='predict_proba')

oof_loss = log_loss(y_train, oof_preds)
print(f'TabPFN OOF log_loss: {oof_loss:.4f}')

In [ ]:
# Fit on all data, predict test
clf.fit(X_train, y_train)
test_preds = clf.predict_proba(X_test)

print(f'Test predictions shape: {test_preds.shape}')
print(f'Row sums: {test_preds.sum(axis=1)[:5]}')

In [ ]:
# Save
np.save(OUTPUT_DIR / 'tabpfn_oof.npy', oof_preds)
np.save(OUTPUT_DIR / 'tabpfn_test.npy', test_preds)
print('Saved TabPFN predictions')